# Dunnhumby seed 48: M1 vs reliability-gated orthogonal N/V M2

- seed 43 특이성을 확인하기 위한 단일 사전 고정 재현 실험입니다.
- M1과 M2를 seed 48에서 모두 처음부터 300 epoch 학습합니다.
- development 684–690만 사용하며 test·holdout은 열지 않습니다.
- 예상 소요 시간은 GPU 상태에 따라 약 3–4시간입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys, torch

SOURCE_COMMIT = 'b541ba781614'
REPO = Path('/content/clv-m2-seed48-' + SOURCE_COMMIT)
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short=12', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


In [ ]:
import clv_reliability_orthogonal_nv_seed48_replication as screen

cfg, prepared = screen.prepare()
# 여기까지는 학습 없음. 다음 셀이 M1 후 M2를 순차 학습합니다.


In [ ]:
result = screen.run(cfg, prepared)
print(result['reading'])


In [ ]:
from IPython.display import display

display(result['absolute'][result['absolute']['epoch'].isin([100, 300])].sort_values(['epoch', 'model_id']))
display(result['comparison'][result['comparison']['epoch'].eq(300)].sort_values('metric'))


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    '/content/clv_reliability_orthogonal_nv_seed48_replication_results',
    'zip',
    str(Path(cfg.out_dir) / 'reports'),
)
files.download(archive)
